In [3]:
import pandas as pd
import numpy as np

# ==========================================================
# CONFIG (edit these paths + thresholds)
# ==========================================================
dejavu_path = "cosines_dejavu.parquet"
fontA_path  = "cosines_unifont.parquet"
fontB_path  = "cosines_gentium.parquet"

# column names (edit if yours differ)
fraud_col = "fraudulent_name"
real_col  = "real_name"
label_col = "label"          # 0/1 ground truth
cos_col   = "cosine_sim"         # cosine similarity for that font

# Youden thresholds (replace with your computed values)
youden_dejavu = 0.8149772286415100
youden_A      = 0.8102402091026306
youden_B      = 0.8164674639701843

# Define "positive" prediction convention.
# Common for cosine-based "match": predict same-class/legit if cosine >= threshold.
# If in your setup it's inverted, flip the comparison below.
def pred_from_cos(cos, thr):
    return (cos >= thr).astype(int)

def signed_margin(cos, thr, y):
    """
    Positive => on the correct side of threshold for label y.
    y=1: want cos >= thr   => margin = cos - thr
    y=0: want cos <  thr   => margin = thr - cos
    """
    return np.where(y == 1, cos - thr, thr - cos)

# ==========================================================
# LOAD (each parquet contains the same rows/order OR has keys to merge)
# ==========================================================
df_d = pd.read_parquet(dejavu_path)
df_a = pd.read_parquet(fontA_path)
df_b = pd.read_parquet(fontB_path)

# ==========================================================
# MERGE SAFELY ON KEYS
# ==========================================================
keys = [fraud_col, real_col, label_col]
for k in keys:
    if k not in df_d.columns or k not in df_a.columns or k not in df_b.columns:
        raise ValueError(f"Missing key column '{k}' in one of the parquets.")

keep_d = keys + [cos_col]
keep_a = keys + [cos_col]
keep_b = keys + [cos_col]

df = (
    df_d[keep_d].rename(columns={cos_col: "cos_dejavu"})
      .merge(df_a[keep_a].rename(columns={cos_col: "cos_A"}), on=keys, how="inner")
      .merge(df_b[keep_b].rename(columns={cos_col: "cos_B"}), on=keys, how="inner")
)

print("Merged shape:", df.shape)

# ==========================================================
# PREDICTIONS
# ==========================================================
df["pred_dejavu"] = pred_from_cos(df["cos_dejavu"].to_numpy(), youden_dejavu)
df["pred_A"]      = pred_from_cos(df["cos_A"].to_numpy(),      youden_A)
df["pred_B"]      = pred_from_cos(df["cos_B"].to_numpy(),      youden_B)

# ==========================================================
# FILTER: DejaVu wrong, A+B right
# ==========================================================
y = df[label_col].astype(int)

mask = (
    (df["pred_dejavu"] != y) &
    (df["pred_A"] == y) &
    (df["pred_B"] == y)
)

hits = df.loc[mask, [fraud_col, real_col, label_col, "cos_dejavu", "cos_A", "cos_B", "pred_dejavu", "pred_A", "pred_B"]].copy()

# ==========================================================
# "BIGGEST CULPRIT" SCORING
#   large => A/B very confidently correct, DejaVu very confidently wrong
# ==========================================================
y_hits = hits[label_col].to_numpy()

hits["m_dejavu"] = signed_margin(hits["cos_dejavu"].to_numpy(), youden_dejavu, y_hits)
hits["m_A"]      = signed_margin(hits["cos_A"].to_numpy(),      youden_A,      y_hits)
hits["m_B"]      = signed_margin(hits["cos_B"].to_numpy(),      youden_B,      y_hits)

hits["m_AB_avg"] = 0.5 * (hits["m_A"] + hits["m_B"])
hits["culprit"]  = hits["m_AB_avg"] - hits["m_dejavu"]

# Top-K worst cases
K = 25
worst = hits.sort_values("culprit", ascending=False).head(K)

display(worst[[fraud_col, real_col, label_col, "culprit", "cos_dejavu", "cos_A", "cos_B", "m_dejavu", "m_A", "m_B"]])

print("Matches:", len(hits))
display(hits[[fraud_col, real_col]])

# (optional) keep debug view
# display(hits)

Merged shape: (10011, 6)


,fraudulent_name,real_name,label,culprit,cos_dejavu,cos_A,cos_B,m_dejavu,m_A,m_B
2119,dles1gnspiretion,designspiration,1.0,0.153675,0.722408,0.914650,0.834268,-0.092570,0.104410,0.017801
6937,ŋŝ5ñ,ns5n,1.0,0.136314,0.783300,0.964640,0.871341,-0.031677,0.154400,0.054873
7612,inloakat5uki,indoakatsuki,1.0,0.127783,0.769138,0.939300,0.851296,-0.045839,0.129059,0.034829
9534,ȕnich4rm,unicharm,1.0,0.127758,0.771596,0.950198,0.845264,-0.043382,0.139957,0.028796
3557,getc2editçardňumbers,getcreditcardnumbers,1.0,0.124974,0.795867,0.951264,0.887171,-0.019110,0.141024,0.070703
49,àutoma4ionđirect,automationdirect,1.0,0.116820,0.789301,0.942638,0.866358,-0.025676,0.132398,0.049890
5385,m4řȑketwireď,marketwired,1.0,0.103748,0.803363,0.906872,0.904104,-0.011614,0.096631,0.087636
437,the-qrcode-ganera7or,the-qrcode-generator,1.0,0.103256,0.812709,0.947875,0.880807,-0.002268,0.137635,0.064339
9084,palijgbis6,palingbisa,1.0,0.093935,0.775262,0.911805,0.823342,-0.039715,0.101565,0.006875
6389,ip3alar,apsalar,1.0,0.092498,0.770865,0.907006,0.816472,-0.044113,0.096765,0.000005


Matches: 138


,fraudulent_name,real_name
49,àutoma4ionđirect,automationdirect
57,tc104,so1024
198,joblaţin,jobnation
212,51you,51din
280,ŕesponsivemiracleeu,responsivemiracle
...,...,...
9635,wųṽm,wuv
9665,wg×ɍ,wqxr
9761,g4virtual5chool,gavirtualschool
9832,p_ra-ov,prao
